# Q3 — Text Summarization (CNN/DailyMail)
**Models:** Lead-3 (CPU) → TextRank (CPU) → BART / DistilBART (GPU)

**3 paradigms:** Lead-3 (naive extractive), TextRank (graph extractive), BART (abstractive)

**Dataset:** 1000 val + 1000 test samples

**Runtime:** T4 GPU required (for BART)

## 1. Environment Setup

In [ ]:
import os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q3'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Drive mounted, output dir ready.')

In [ ]:
REPO_DIR = '/content/467-takehome'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Run Lead-3 + TextRank (CPU)

In [ ]:
!python -m src.q3_summarization.main \
    --config configs/q3.yaml \
    --final-eval \
    --override \
        models.textrank.enabled=true \
        models.lead3.enabled=true \
        models.bart.enabled=false \
        dataset.limit_train_samples=10000 \
        dataset.limit_validation_samples=1000 \
        dataset.limit_test_samples=1000

In [ ]:
import glob, shutil
latest_run = sorted(glob.glob('outputs/q3/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_extractive')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'Extractive results saved to Drive: {dest}')

## 3. Run BART (GPU)

In [ ]:
!python -m src.q3_summarization.main \
    --config configs/q3.yaml \
    --final-eval \
    --override \
        models.textrank.enabled=false \
        models.lead3.enabled=false \
        models.bart.enabled=true \
        models.bart.model_name=sshleifer/distilbart-cnn-12-6 \
        models.bart.batch_size=4 \
        models.bart.max_input_length=1024 \
        models.bart.max_output_length=142 \
        models.bart.min_output_length=56 \
        dataset.limit_train_samples=10000 \
        dataset.limit_validation_samples=1000 \
        dataset.limit_test_samples=1000

In [ ]:
latest_run = sorted(glob.glob('outputs/q3/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_bart')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'BART results saved to Drive: {dest}')

## 4. Results Summary

In [ ]:
import json

print('=== Q3 Results Summary ===')
for run_dir in sorted(glob.glob('outputs/q3/run_*')):
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            metrics = json.load(f)
        print(f'\n--- {os.path.basename(run_dir)} ---')
        print(json.dumps(metrics, indent=2, default=str)[:2000])